# Step 10 — Constitutional Dimension Trends Over Time

How have the 14 constitutional dimensions changed across history?

**Approach:**
- Compute the global average score for each dimension by year (and by decade)
- Line chart: all 14 dimensions over time
- Identify which dimensions have changed the most (by absolute swing and by trend slope)
- Heatmap of decade-level averages for a cleaner overview

**Input:** `ccpc_axis_scores_llm.csv`  
**Output:** figures in `outputs/10_dimension_trends/`

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from scipy import stats
import os

os.makedirs('outputs/10_dimension_trends', exist_ok=True)
print('Libraries loaded.')

Libraries loaded.


In [2]:
SCORES_PATH = 'ccpc_axis_scores_llm.csv'

DIMS = [
    'ccpc_civil_liberties',
    'ccpc_socioeconomic_rights',
    'ccpc_political_competition',
    'ccpc_legislative_autonomy',
    'ccpc_executive_constraints',
    'ccpc_judicial_independence',
    'ccpc_rule_of_law_due_process',
    'ccpc_institutional_accountability',
    'ccpc_emergency_powers_constraints',
    'ccpc_civilian_control_of_security',
    'ccpc_amendment_rigidity',
    'ccpc_federalism_decentralization',
    'ccpc_transparency_information_access',
    'ccpc_equality_gender_minority_indigenous',
]

LABELS = {
    'ccpc_civil_liberties':                      'Civil Liberties',
    'ccpc_socioeconomic_rights':                 'Socioeconomic Rights',
    'ccpc_political_competition':                'Political Competition',
    'ccpc_legislative_autonomy':                 'Legislative Autonomy',
    'ccpc_executive_constraints':                'Executive Constraints',
    'ccpc_judicial_independence':                'Judicial Independence',
    'ccpc_rule_of_law_due_process':              'Rule of Law / Due Process',
    'ccpc_institutional_accountability':         'Institutional Accountability',
    'ccpc_emergency_powers_constraints':         'Emergency Powers Constraints',
    'ccpc_civilian_control_of_security':         'Civilian Control of Security',
    'ccpc_amendment_rigidity':                   'Amendment Rigidity',
    'ccpc_federalism_decentralization':          'Federalism / Decentralization',
    'ccpc_transparency_information_access':      'Transparency / Info Access',
    'ccpc_equality_gender_minority_indigenous':  'Equality (Gender/Minority/Indigenous)',
}

## 1 — Load and Prepare Data

In [3]:
df = pd.read_csv(SCORES_PATH)
df['year'] = pd.to_numeric(df['year'], errors='coerce')

# Focus on post-1900 for readability; earlier data is very sparse
df = df[df['year'] >= 1900].copy()

# Decade column
df['decade'] = (df['year'] // 10 * 10).astype(int)

print(f'Rows: {len(df):,}  |  Years: {df["year"].min():.0f}–{df["year"].max():.0f}')
print(f'Countries: {df["cowcode"].nunique()}')

Rows: 14,265  |  Years: 1900–2023
Countries: 202


## 2 — Annual and Decade Averages

In [4]:
# Annual means (smoothed with 5-year rolling average for the line chart)
annual = df.groupby('year')[DIMS].mean()
annual_smooth = annual.rolling(window=5, center=True, min_periods=3).mean()

# Decade means
decade = df.groupby('decade')[DIMS].mean()

print('Decade averages (first 3 dimensions shown):')
print(decade[DIMS[:3]].round(3))

Decade averages (first 3 dimensions shown):
        ccpc_civil_liberties  ccpc_socioeconomic_rights  \
decade                                                    
1900                   0.391                      0.112   
1910                   0.383                      0.113   
1920                   0.395                      0.147   
1930                   0.398                      0.167   
1940                   0.426                      0.227   
1950                   0.445                      0.293   
1960                   0.433                      0.259   
1970                   0.447                      0.284   
1980                   0.467                      0.307   
1990                   0.519                      0.356   
2000                   0.544                      0.380   
2010                   0.562                      0.409   
2020                   0.565                      0.417   

        ccpc_political_competition  
decade                           

## 3 — Which Dimensions Changed Most?

In [5]:
# Use decade data: compare 1900s vs most recent complete decade
first_decade = decade.iloc[0]
last_decade  = decade.iloc[-1]

change = pd.DataFrame({
    'dimension': [LABELS[d] for d in DIMS],
    'start':     [first_decade[d] for d in DIMS],
    'end':       [last_decade[d]  for d in DIMS],
})
change['absolute_change'] = change['end'] - change['start']
change = change.sort_values('absolute_change', ascending=False)

# OLS trend slope per dimension (post-1950 only, more stable coverage)
post50 = annual[annual.index >= 1950].dropna()
slopes = {}
for d in DIMS:
    valid = post50[d].dropna()
    if len(valid) > 10:
        slope, _, _, _, _ = stats.linregress(valid.index, valid.values)
        slopes[LABELS[d]] = slope * 10  # change per decade

change['slope_per_decade'] = change['dimension'].map(slopes)

print(f'Comparing {decade.index[0]}s → {decade.index[-1]}s')
print()
print(change[['dimension','start','end','absolute_change','slope_per_decade']]
      .to_string(index=False))

Comparing 1900s → 2020s

                            dimension    start      end  absolute_change  slope_per_decade
                 Socioeconomic Rights 0.111698 0.417240         0.305542          0.023597
Equality (Gender/Minority/Indigenous) 0.449240 0.644757         0.195518          0.017391
                      Civil Liberties 0.390635 0.565093         0.174459          0.022591
            Rule of Law / Due Process 0.336120 0.508065         0.171945          0.024360
         Institutional Accountability 0.290704 0.450328         0.159624          0.021083
           Transparency / Info Access 0.287003 0.440836         0.153833          0.026255
                Political Competition 0.550526 0.667847         0.117321          0.008894
                Judicial Independence 0.436232 0.540982         0.104751          0.009802
                   Amendment Rigidity 0.600391 0.632246         0.031855          0.008156
                Executive Constraints 0.414441 0.436682         0

## 4 — Line Chart: All 14 Dimensions Over Time

In [6]:
# Sort dimensions by most-changed (ascending absolute change) for legend order
dims_sorted = change.sort_values('absolute_change', ascending=False)['dimension'].tolist()
dims_col_sorted = [d for label in dims_sorted for d in DIMS if LABELS[d] == label]

# Color palette: diverging so rising vs falling dimensions are visually distinct
n = len(DIMS)
palette = plt.cm.tab20(np.linspace(0, 1, n))

fig, ax = plt.subplots(figsize=(12, 6))

for i, dim in enumerate(dims_col_sorted):
    series = annual_smooth[dim].dropna()
    ax.plot(series.index, series.values,
            label=LABELS[dim],
            color=palette[i],
            linewidth=1.6,
            alpha=0.85)

# Annotate key historical periods
for xval, label in [(1918, 'WWI end'), (1945, 'WWII end'),
                    (1989, 'Cold War end'), (2011, 'Arab Spring')]:
    ax.axvline(xval, color='#aaaaaa', lw=0.8, ls='--')
    ax.text(xval + 0.5, 0.02, label, fontsize=7, color='#666666', rotation=90, va='bottom')

ax.set_xlabel('Year', fontsize=10)
ax.set_ylabel('Average Dimension Score (0–1)', fontsize=10)
ax.set_title('How Constitutional Dimensions Have Changed Over Time\n'
             '(5-year rolling average of global mean scores, post-1900)', fontsize=11)
ax.set_xlim(1900, 2025)
ax.set_ylim(0, 1)
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=7.5, frameon=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('outputs/10_dimension_trends/dimension_trends_line.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: outputs/10_dimension_trends/dimension_trends_line.png')

Saved: outputs/10_dimension_trends/dimension_trends_line.png


## 5 — Heatmap: Decade-Level Averages

In [7]:
# Build heatmap matrix: rows = dimensions (sorted by total change), cols = decades
heatmap_data = decade[dims_col_sorted].T
heatmap_data.index = [LABELS[d] for d in dims_col_sorted]

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    heatmap_data,
    ax=ax,
    cmap='RdYlGn',
    vmin=0, vmax=1,
    linewidths=0.4,
    linecolor='white',
    cbar_kws={'label': 'Avg Score (0–1)', 'shrink': 0.6},
    annot=False,
)
ax.set_xlabel('Decade', fontsize=10)
ax.set_ylabel('')
ax.set_title('Constitutional Dimension Scores by Decade\n'
             '(rows sorted by largest absolute change, 1900s → 2020s)', fontsize=11)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', labelsize=8.5)

plt.tight_layout()
plt.savefig('outputs/10_dimension_trends/dimension_trends_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: outputs/10_dimension_trends/dimension_trends_heatmap.png')

Saved: outputs/10_dimension_trends/dimension_trends_heatmap.png


## 6 — Bar Chart: Most Changed Dimensions

In [8]:
fig, ax = plt.subplots(figsize=(8, 5))

colors = ['#2166ac' if v >= 0 else '#d6604d' for v in change['absolute_change']]
bars = ax.barh(
    change['dimension'],
    change['absolute_change'],
    color=colors,
    edgecolor='white',
    height=0.6,
)

ax.axvline(0, color='#333333', lw=0.8)
ax.set_xlabel(f'Change in avg score ({decade.index[0]}s → {decade.index[-1]}s)', fontsize=10)
ax.set_title('Which Constitutional Dimensions Changed Most?\n'
             '(Blue = increased, Red = decreased)', fontsize=11)

for bar, val in zip(bars, change['absolute_change']):
    xpos = val + 0.003 if val >= 0 else val - 0.003
    ha   = 'left' if val >= 0 else 'right'
    ax.text(xpos, bar.get_y() + bar.get_height() / 2,
            f'{val:+.3f}', va='center', ha=ha, fontsize=8)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='y', labelsize=8.5)

plt.tight_layout()
plt.savefig('outputs/10_dimension_trends/dimension_change_bar.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved: outputs/10_dimension_trends/dimension_change_bar.png')

print(f'\nSummary: Decades compared: {decade.index[0]}s → {decade.index[-1]}s')
print(f'Most increased: {change.iloc[0]["dimension"]} ({change.iloc[0]["absolute_change"]:+.3f})')
print(f'Most decreased: {change.iloc[-1]["dimension"]} ({change.iloc[-1]["absolute_change"]:+.3f})')

Saved: outputs/10_dimension_trends/dimension_change_bar.png

Summary: Decades compared: 1900s → 2020s
Most increased: Socioeconomic Rights (+0.306)
Most decreased: Federalism / Decentralization (-0.030)
